# 07 · As a Tool — 03 a second tool

**Runs end to end with no API key.** The tool choice below is made by a deterministic stand-in chooser unless `GROQ_API_KEY` or `OPENAI_API_KEY` is loaded, in which case one optional cell asks a real model to make the same choice under a spend ceiling. `nbio.show_environment()` in Step 2 says which one you are reading.

Stage `02` collapsed six stages of retrieval into one function. This notebook adds a second function that has no retrieval in it at all — it subtracts two dates — and then asks a question with both tools on the table.

**This is where the repo stops being RAG and starts being an agent: with one tool there is no choice to make; with two there is, and something has to make it.**

That is the whole content of the word "agent" at this point. Not autonomy, not planning, not a framework: a choice point that did not exist one cell earlier. Everything the rest of this repo does — guardrails, orchestration, observability — exists because of the thing that appears in Step 7.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `search_documents` | Tool one: the retrieval pipeline of stages `01`–`06`, behind one schema. | `search_documents("What is the refund policy...", k=2)` -> handbook passages |
| `days_between` | Tool two: subtracts two dates. No retrieval, no corpus, no model. | `days_between("2026-08-02", "2026-09-04")` -> `33` |
| `route` | Picks one tool from the specs available, by overlap with the question. | `route(q, specs)` -> `("days_between", {...args})` |
| The one-tool case | With a single spec, every question routes the same way — there is no decision. | 3 unrelated questions -> `search_documents` every time |
| `run_agent` | question -> choose a tool -> build arguments -> call it -> answer. | `run_agent("How many days between ...")` |
| The question that needs both | A single-choice router structurally cannot answer it. Named, not hidden. | shipped `2026-08-02`, returned `2026-09-04`, 30-day policy |

## Step 1 — bootstrap the repo path

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

## Step 2 — which path this run is on

Steps 7 and 8 produce a tool choice either way. This readout is how you know whether the choice you are about to read came from a model or from the deterministic stand-in.

In [ ]:
nbio.show_environment()

## Step 3 — tool one: the retrieval pipeline, as one function

The same `search_documents` contract as `02-pipeline-as-tool.ipynb`, over a four-passage company handbook written for this notebook. Extraction and chunking are already done here — the passages are the chunks — because this notebook is about what happens *around* the tool, not inside it. The embedding is the same deterministic offline hash embedding; the score floor is the same illustrative value.

In [ ]:
import hashlib
import math

DIM = 256
SCORE_FLOOR = 0.08  # illustrative, not a tuned value

HANDBOOK = [
    ("returns-policy.md", "Items may be returned within 30 days of the shipping date for a full "
                          "refund, provided they are unused and in original packaging."),
    ("returns-policy.md", "Refunds are issued to the original payment method within 5 business days "
                          "of the returned item arriving at the warehouse."),
    ("shipping.md", "Standard shipping takes 3-5 business days; expedited shipping takes 1-2 "
                    "business days once an order leaves the warehouse."),
    ("warranty.md", "Electronics carry a one-year limited warranty against manufacturing defects, "
                    "which is separate from the return window."),
]


def hash_embed(text: str, dim: int = DIM) -> list[float]:
    """Deterministic, offline embedding -- no model, no API key, no network."""
    vec = [0.0] * dim
    for tok in (text or "").lower().split():
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        vec[h % dim] += 1.0 if (h >> 8) & 1 else -1.0
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


INDEX = [
    {"chunk_id": f"{src}#{i}", "source": src, "text": text, "embedding": hash_embed(text)}
    for i, (src, text) in enumerate(HANDBOOK)
]


def search_documents(query: str, k: int = 3) -> list[dict]:
    """Search the company handbook and return the passages most relevant to a question about returns, refunds, shipping, or warranty coverage.

    Args:
        query: The question to search for, in natural language.
        k: How many passages to return, most relevant first.
    """
    qvec = hash_embed(query)
    scored = [
        {"chunk_id": c["chunk_id"], "source": c["source"], "text": c["text"],
         "score": round(sum(x * y for x, y in zip(qvec, c["embedding"])), 4)}
        for c in INDEX
    ]
    scored.sort(key=lambda c: -c["score"])
    return [c for c in scored[:k] if c["score"] >= SCORE_FLOOR]


print(f"{len(INDEX)} handbook passages indexed")
for hit in search_documents("What is the refund policy for an unused item?", k=2):
    print(f"  {hit['score']:.4f}  {hit['chunk_id']}  {hit['text'][:62]}...")

## Step 4 — tool two: deliberately trivial, and deliberately not retrieval

`days_between` subtracts two dates. There is no corpus behind it, no embedding, no model, nothing to rank, and nothing that could plausibly be confused with search. That is the point: the second tool has to be a genuinely different *kind* of thing for the choice in Step 7 to be a real choice rather than a tie-break between two flavours of the same capability.

It is also the shape of tool that turns out to matter most in practice. A model is bad at arithmetic on dates and good at deciding that arithmetic on dates is what is needed.

In [ ]:
from datetime import date


def days_between(start_date: str, end_date: str) -> int:
    """Count the number of whole calendar days between two dates, each written as YYYY-MM-DD.

    Args:
        start_date: The earlier date, as YYYY-MM-DD.
        end_date: The later date, as YYYY-MM-DD.
    """
    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)
    return (end - start).days


print(f"days_between('2026-08-02', '2026-09-04') = {days_between('2026-08-02', '2026-09-04')}")
assert days_between("2026-08-02", "2026-09-04") == 33
assert days_between("2026-01-01", "2026-01-01") == 0
assert days_between("2026-09-04", "2026-08-02") == -33, "order matters, and the sign says so"
print("a date subtraction, nothing more -- no corpus, no embedding, no model")

## Step 5 — both functions, as specs

The same spec builder as `01-function-as-tool.ipynb`, inlined so this notebook stands on its own. Two names, two descriptions, two argument schemas. This list is the entire universe the chooser gets to reason over.

In [ ]:
import inspect
import json
import re
import typing

_JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}


def tool_spec(fn) -> dict:
    """The name, description and argument schema a model is shown for one function."""
    doc = inspect.getdoc(fn) or ""
    head, arg_docs, in_args = [], {}, False
    for line in doc.splitlines():
        stripped = line.strip()
        if stripped.lower().rstrip(":") in {"args", "arguments", "parameters"}:
            in_args = True
            continue
        if in_args:
            if stripped and ":" in stripped:
                name, _, desc = stripped.partition(":")
                arg_docs[name.strip()] = desc.strip()
        else:
            head.append(stripped)

    hints = typing.get_type_hints(fn)
    properties, required = {}, []
    for name, param in inspect.signature(fn).parameters.items():
        prop = {"type": _JSON_TYPES.get(hints.get(name, str), "string")}
        if name in arg_docs:
            prop["description"] = arg_docs[name]
        if param.default is inspect.Parameter.empty:
            required.append(name)
        else:
            prop["default"] = param.default
        properties[name] = prop

    return {
        "name": fn.__name__,
        "description": " ".join(p for p in head if p).strip(),
        "parameters": {"type": "object", "properties": properties, "required": required},
    }


TOOLS = {fn.__name__: {"fn": fn, "spec": tool_spec(fn)} for fn in (search_documents, days_between)}
SPECS = [entry["spec"] for entry in TOOLS.values()]

print(json.dumps(SPECS, indent=2))
assert len(SPECS) == 2

## Step 6 — the chooser, and the argument builder

Two jobs a model does in one step, split apart here so both are visible:

1. **Which tool** — word overlap between the question and each spec's name and description, with a crude singular/plural fold. Not a model; the same stand-in `01-function-as-tool.ipynb` used.
2. **Which arguments** — two ISO dates pulled out of the question with a regex for `days_between`, the question itself as the `query` for `search_documents`.

The second half is the cruder of the two by a wide margin. A real model returns arguments as JSON, generated from the schema, for any phrasing of a date; this regex handles `YYYY-MM-DD` and nothing else. It is here so that the loop in Step 8 is a real loop with real arguments rather than a hand-fed one.

In [ ]:
STOPWORDS = {
    "a", "an", "and", "are", "at", "be", "by", "can", "did", "do", "does", "for",
    "from", "has", "have", "how", "i", "in", "is", "it", "many", "me", "my", "of",
    "on", "our", "that", "the", "this", "to", "too", "what", "when", "which", "who",
    "will", "with", "you", "your",
}


def _tokens(text: str) -> set[str]:
    """Words, lowercased, minus stopwords, with a trailing plural -s folded away."""
    out = set()
    for word in re.findall(r"[a-z]+", (text or "").lower()):
        if word in STOPWORDS:
            continue
        out.add(word[:-1] if len(word) > 3 and word.endswith("s") else word)
    return out


def build_arguments(tool_name: str, question: str) -> dict:
    """The arguments a model would have generated from the schema. A regex stands in."""
    if tool_name == "days_between":
        found = re.findall(r"\d{4}-\d{2}-\d{2}", question)
        return {"start_date": found[0], "end_date": found[1]} if len(found) >= 2 else {}
    return {"query": question}


def route(question: str, specs: list[dict]) -> tuple[str, dict, list[tuple[str, int]]]:
    """Choose one tool and its arguments. Returns (name, arguments, the full score table)."""
    q = _tokens(question)
    scored = [(s["name"], len(_tokens(s["name"] + " " + s["description"]) & q)) for s in specs]
    scored.sort(key=lambda pair: (-pair[1], pair[0]))  # ties break alphabetically, reproducibly
    chosen = scored[0][0]
    return chosen, build_arguments(chosen, question), scored

## Step 7 — one tool: there is no decision here

Run the router over three questions that have nothing to do with each other, with only `search_documents` on the table. Every one of them routes to `search_documents` — not because the router judged well, but because there was nothing else to return. This is RAG: a fixed pipeline with a query on the front of it. The router is a formality.

In [ ]:
QUESTIONS = [
    "What is the refund policy for an unused item?",
    "How many days are there between 2026-08-02 and 2026-09-04?",
    "My order shipped on 2026-08-02. Is it too late to return it on 2026-09-04?",
]

one_tool_specs = [TOOLS["search_documents"]["spec"]]
one_tool_picks = [route(q, one_tool_specs)[0] for q in QUESTIONS]


def _short(text: str, width: int = 52) -> str:
    return text if len(text) <= width else text[:width] + "..."


nbio.table(
    [(_short(q), pick) for q, pick in zip(QUESTIONS, one_tool_picks)],
    headers=("question", "routed to"),
)
print()
print(f"distinct tools reachable : {len(set(one_tool_picks))}")
print(f"decisions actually made  : 0")

assert len(set(one_tool_picks)) == 1, "with one tool there is only one possible outcome"
assert set(one_tool_picks) == {"search_documents"}

## Step 8 — two tools: the decision exists, and something has to make it

The same three questions, the same router, one more entry in the spec list. Question two now routes somewhere else — and nothing about the router changed. The choice appeared because the second tool appeared.

With one tool there is no choice to make; with two there is, and something has to make it. In this notebook that something is 12 lines of word overlap. In a real agent it is a model, and every failure mode this repo's remaining stages exist to handle — the wrong tool, the right tool with wrong arguments, a tool that should have been called twice — starts at exactly this line.

In [ ]:
rows = []
for question in QUESTIONS:
    chosen, args, scores = route(question, SPECS)
    score_text = ", ".join(f"{name}={n}" for name, n in scores)
    rows.append((_short(question), chosen, score_text))

nbio.table(rows, headers=("question", "routed to", "overlap scores"))

two_tool_picks = [route(q, SPECS)[0] for q in QUESTIONS]
print()
print(f"one tool available  -> picks: {one_tool_picks}")
print(f"two tools available -> picks: {two_tool_picks}")

assert one_tool_picks[1] == "search_documents"
assert two_tool_picks[1] == "days_between", "the date question should reach the date tool"
assert len(set(two_tool_picks)) == 2, "two tools, and both of them get used"

## Step 9 — the loop, end to end

Question in, answer out: choose a tool, build its arguments, call the real function, print what came back. Twenty lines, no framework. A framework would give you retries, tracing, schema validation and a message history — and it would do this in the middle.

In [ ]:
def run_agent(question: str) -> dict:
    """Route a question to one tool, call it, and return what happened."""
    chosen, args, _ = route(question, SPECS)
    result = TOOLS[chosen]["fn"](**args)
    return {"question": question, "tool": chosen, "arguments": args, "result": result}


for question in QUESTIONS[:2]:
    step = run_agent(question)
    print(f"question  : {step['question']}")
    print(f"tool      : {step['tool']}")
    print(f"arguments : {step['arguments']}")
    if isinstance(step["result"], list):
        for hit in step["result"]:
            print(f"result    : [{hit['score']}] {hit['text'][:70]}...")
    else:
        print(f"result    : {step['result']}")
    print()

date_step = run_agent(QUESTIONS[1])
assert date_step["tool"] == "days_between"
assert date_step["result"] == 33, "the tool did the arithmetic, not the router"

## Step 10 — the third question, which one choice cannot answer

`"My order shipped on 2026-08-02. Is it too late to return it on 2026-09-04?"` needs the handbook (what is the return window?) **and** the date tool (how wide is this gap?). The router picks exactly one tool, so it returns the policy and stops — a true, useless half-answer. Nothing in this notebook is capable of noticing that.

The cell below then does what the router could not: calls both tools and combines them. Read the combining code carefully, because it is hand-written — a human decided the order, decided that a number of days should be pulled out of the retrieved passage, and decided what to compare it to. Deciding that sequence automatically is orchestration, and it is stage `04-orchestrate`, not this one.

Watch the top-ranked passage while you are there. The hash embedding puts the *shipping times* passage above the *return window* passage, because it ranks shared vocabulary and both passages are full of the same words. The composing code has to scan the shortlist rather than trust rank one — which is another hand-written decision, made necessary by a weakness one stage upstream.

In [ ]:
hard_question = QUESTIONS[2]
routed_tool, routed_args, _ = route(hard_question, SPECS)
half_answer = run_agent(hard_question)

print(f"question    : {hard_question}")
print(f"routed to   : {routed_tool}  (one tool, one call)")
print(f"what it got : {half_answer['result'][0]['text'] if half_answer['result'] else '(nothing)'}")
print()
print("-- both tools, composed by hand --")

policy_hits = search_documents("return window after the shipping date", k=3)
print(f"top-ranked passage : {policy_hits[0]['text'][:66]}...")

# Rank 1 is about shipping times, not the return window -- so the composing code
# scans the shortlist for a passage that states a number of days instead of
# trusting the top hit.
window_days, window_hit = None, None
for hit in policy_hits:
    match = re.search(r"(\d+)\s+days", hit["text"])
    if match:
        window_days, window_hit = int(match.group(1)), hit
        break

elapsed = days_between("2026-08-02", "2026-09-04")
too_late = elapsed > window_days

print(f"passage used       : {window_hit['text'][:66]}...")
print(f"search_documents -> policy window : {window_days} days")
print(f"days_between     -> elapsed       : {elapsed} days")
print(f"combined answer  -> too late?     : {too_late}")

assert routed_tool == "search_documents", "one choice, and it is the retrieval half"
assert window_days == 30 and elapsed == 33
assert too_late is True
assert policy_hits[0] is not window_hit, "the passage that answered was not the top-ranked one"
print()
print("The router made one choice and got half the answer. The other half came from")
print("a second call this notebook has no mechanism to decide on -- that mechanism is")
print("stage 04-orchestrate.")

## Step 11 — the same two specs, given to a real model — only if a key is loaded

Everything above was decided by word overlap. If a key is loaded, this cell hands the same two specs and the same three questions to a real model and prints which tool it asked for and with which arguments — the honest comparison against the stand-in, under a spend ceiling. With no key it says so and skips; the deterministic result above already made the argument.

In [ ]:
import os


def _client():
    if os.environ.get("GROQ_API_KEY"):
        from groq import Groq

        return "groq", Groq(api_key=os.environ["GROQ_API_KEY"]), "llama-3.3-70b-versatile"
    if os.environ.get("OPENAI_API_KEY"):
        from openai import OpenAI

        return "openai", OpenAI(api_key=os.environ["OPENAI_API_KEY"]), "gpt-4o-mini"
    return None, None, None


provider, client, model_id = _client()

with nbio.cost_meter(budget_usd=0.50) as meter:
    if provider is None:
        print(
            "No GROQ_API_KEY or OPENAI_API_KEY loaded -- not set, skipping. Running the "
            "deterministic stand-in instead: every tool choice printed above came from "
            "`route()`, which is word overlap, not a model."
        )
    else:
        print(f"provider={provider!r} model={model_id!r}")
        for question in QUESTIONS:
            resp = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": question}],
                tools=[{"type": "function", "function": s} for s in SPECS],
                temperature=0.0,
            )
            usage = getattr(resp, "usage", None)
            meter.record(
                model_id,
                getattr(usage, "prompt_tokens", 0) if usage else 0,
                getattr(usage, "completion_tokens", 0) if usage else 0,
            )
            calls = resp.choices[0].message.tool_calls or []
            stand_in = route(question, SPECS)[0]
            model_choice = ", ".join(c.function.name for c in calls) or "(answered without a tool)"
            print(f"  q         : {question[:60]}")
            print(f"  stand-in  : {stand_in}")
            print(f"  model     : {model_choice}")
            for call in calls:
                print(f"  arguments : {call.function.arguments}")

print()
print(meter.report())

## What did not come across

- **The stand-in is not a model, and the gap is widest exactly here.** Word overlap picks a tool from vocabulary. A model picks from meaning, can ask for *two* tools in one response (which is how question three is really answered), and can rewrite the query before searching. Step 11 is the cell that shows you the difference, and it needs a key.
- **No message history.** A real loop appends the tool's result to the conversation and calls the model again, so it can react to what came back. Here the result is printed and the loop ends. That second turn is where "agent" starts meaning something more than "router".
- **No argument validation.** `build_arguments` can return `{}` for a date question with only one date in it, and `run_agent` would call `days_between()` with missing arguments and raise. That is deliberate — it is the first case `04-tool-failure.ipynb` picks up.
- **Two tools is the smallest interesting number, not a realistic one.** Choice quality degrades as tools are added and their descriptions start to overlap; nothing here measures that, and at two tools with disjoint vocabularies it cannot.

Next: `04-tool-failure.ipynb` — what the loop above should do when the tool it chose raises, hangs, or returns something confidently wrong.